In [ ]:
import pandas as pd 
import json
import pickle
import numpy as np

# Compatibility shim for libraries that still reference removed NumPy aliases.
if not hasattr(np, "NaN"):
    np.NaN = np.nan
if not hasattr(np, "Inf"):
    np.Inf = np.inf

# import own Python files, vars, mappings, and functions
from config import (NAME_REF_DB, COST_DATA_PTX, PROJECT_NAME, NAME_FUTURE_DB,
                   FUTURE_POWER_PRICES, LOCATIONS, FILE_PATH_CASE_STUDIES_PTX,
                   FILE_PATH_CASE_STUDIES_LCA_PTX, OUTPUT_FILE_XARRAY)

import opt_ptx_functions as opt_ptx
import bw2data
import time
import energy_data_processor as ep
from energy_data_processor import country_to_iso2

bw2data.projects.set_current(PROJECT_NAME)

COST_DICT = COST_DATA_PTX[NAME_REF_DB].to_dict() # techno-economic data
COST_DICT_FUTURE = COST_DATA_PTX[NAME_FUTURE_DB].to_dict() # future techno-economic data
gen_results=True

with open(OUTPUT_FILE_XARRAY, "rb") as f:
    ds_processed = pickle.load(f)

# Read GHG info for system components
with open("input_data/dict_ghg_impacts_ptx.txt", 'r') as file:
    DICT_GHG_IMPACTS = json.load(file)

with open("input_data/dict_ghg_impacts_ptx_future.txt", 'r') as file:
    DICT_GHG_IMPACTS_FUTURE = json.load(file)

file_path = FILE_PATH_CASE_STUDIES_PTX
file_path_lca = FILE_PATH_CASE_STUDIES_LCA_PTX
DICT_GHG_IMPACTS

In [ ]:
# load future power prices
FUTURE_POWER_PRICES_FUTURE_DF = pd.read_excel(
    FUTURE_POWER_PRICES,
    sheet_name="lcoe",
    index_col="country",
)[["2 degree_2050"]].copy()

def safe_country_to_iso2(country_name):
    try:
        return country_to_iso2(country_name)
    except Exception:
        return None

FUTURE_POWER_PRICES_FUTURE_DF["iso2"] = (
    FUTURE_POWER_PRICES_FUTURE_DF.index.to_series().map(safe_country_to_iso2)
)

# drop countries that couldn't be mapped
missing_countries = FUTURE_POWER_PRICES_FUTURE_DF[
    FUTURE_POWER_PRICES_FUTURE_DF["iso2"].isna()
].index.tolist()

if missing_countries:
    print("Skipped countries with no ISO2 match:", missing_countries)

FUTURE_POWER_PRICES_FUTURE_DF = FUTURE_POWER_PRICES_FUTURE_DF.dropna(subset=["iso2"])
FUTURE_POWER_PRICES_FUTURE_DF = FUTURE_POWER_PRICES_FUTURE_DF.reset_index()
FUTURE_POWER_PRICES_FUTURE_DF = FUTURE_POWER_PRICES_FUTURE_DF[["iso2", "2 degree_2050"]]
FUTURE_POWER_PRICES_FUTURE_DF = FUTURE_POWER_PRICES_FUTURE_DF.set_index("iso2")

FUTURE_POWER_PRICES_DICT = FUTURE_POWER_PRICES_FUTURE_DF["2 degree_2050"].to_dict()
FUTURE_POWER_PRICES_DICT


In [ ]:
# Optional: Limit locations for faster testing
# LOCATIONS = LOCATIONS[:2]

# PtX pathway selection (loop over all)
PTX_PATHWAYS = ["meoh", "meoh_to_saf", "ftsaf"]

# Performance configuration
CALC_ALL_LCA_IMPACTS = True  # Set to False to skip detailed LCA calculations

# Solver settings (PtX problems are larger than NH3, need more time)
SOLVER_TIME_LIMIT = 3600      # 1 hour (3600s)
SOLVER_MIP_GAP = 0.05         # 5% gap (relaxed for faster convergence)
SOLVER_INT_FEAS_TOL = 1e-6    # Relaxed tolerance
SOLVER_THREADS = 6            # Use 6 physical cores for Gurobi

print("="*80)
print("CONFIGURATION CHECK:")
print("="*80)
print(f"Locations: {len(LOCATIONS)}")
print(f"PTX pathways: {PTX_PATHWAYS}")
print(f"LCA calculations: {'ENABLED' if CALC_ALL_LCA_IMPACTS else 'DISABLED'}")
print(f"Solver time limit: {SOLVER_TIME_LIMIT}s ({SOLVER_TIME_LIMIT/3600:.2f}h)")
print(f"Solver MIP gap: {SOLVER_MIP_GAP*100}%")
print(f"Solver threads: {SOLVER_THREADS}")
print("="*80)


In [ ]:
# Define scenarios
scenarios = {
    "grid_connected": {},
    "hybrid": {},
    "hybrid-green": {"hybrid_green": True, "ghg_reduction": 0.6},
    "off_grid": {},
}

# Grid cap reduction for hybrid-green (fraction of max_grid_cap)
GREEN_GRID_CAP_FACTOR = 0.2

# Fossil baseline GHG intensities (tCO2-eq per t product) for hybrid-green cap
# These are used with ghg_reduction to set: cap = production * baseline * (1 - reduction)
FOSSIL_BASELINE_TCO2 = {
    "meoh": 0.7,          # fossil methanol from natural gas
    "meoh_to_saf": 3.9,   # fossil kerosene (well-to-wake)
    "ftsaf": 3.9,          # fossil kerosene (well-to-wake)
}

print(f"Running {len(scenarios)} scenario(s): {list(scenarios.keys())}")
print(f"Fossil baselines (tCO2/t product): {FOSSIL_BASELINE_TCO2}")



In [ ]:
from config import T_DAY_MEOH, T_DAY_SAF

# Valid parameters for PtX optimization functions
VALID_PTX_PARAMS = {
    "sec_db",
    "calc_all_lca_impacts",
    "export_alias",
    "loc_elect",
    "size_product_system",
    "logger",
    "time_limit",
    "mip_gap",
    "int_feas_tol",
    "iis_path",
    "grid_inj",
    "credit_env_export",
    "autonomous_elect",
    "no_renewables",
    "heuristics",
    "h2_price",
    "euro_ton_co2",
    "eps_ghg_constraint",
    "hybrid_green",
    "ghg_baseline_tco2_per_tprod",
    "ghg_reduction",
    "consider_down_times",
    "export_results",
    "threads",
}

def run_ptx_opt(
    df_data,
    w_cost,
    w_env,
    cost_dict,
    dict_ghg_impacts,
    dict_limits,
    scenario_name,
    loc_elect,
    ptx_pathway,
    **kwargs,
):
    """
    Wrapper for PtX optimization functions.
    Filters out invalid kwargs before passing to optimizer.
    """
    size_product_system = (T_DAY_MEOH if ptx_pathway == "meoh" else T_DAY_SAF) * 365

    filtered_kwargs = {k: v for k, v in kwargs.items() if k in VALID_PTX_PARAMS}

    if ptx_pathway == "meoh":
        return opt_ptx.opt_dac_pem_meoh(
            df_data, w_cost, w_env, cost_dict, dict_ghg_impacts, dict_limits,
            export_alias=scenario_name,
            loc_elect=loc_elect,
            size_product_system=size_product_system,
            **filtered_kwargs,
        )

    if ptx_pathway == "meoh_to_saf":
        return opt_ptx.opt_dac_pem_meoh_to_saf(
            df_data, w_cost, w_env, cost_dict, dict_ghg_impacts, dict_limits,
            export_alias=scenario_name,
            loc_elect=loc_elect,
            size_product_system=size_product_system,
            **filtered_kwargs,
        )

    if ptx_pathway == "ftsaf":
        return opt_ptx.opt_dac_pem_ftsaf(
            df_data, w_cost, w_env, cost_dict, dict_ghg_impacts, dict_limits,
            export_alias=scenario_name,
            loc_elect=loc_elect,
            size_product_system=size_product_system,
            **filtered_kwargs,
        )

    raise ValueError(f"Unknown ptx_pathway: {ptx_pathway}")


In [ ]:
PTX_PATHWAYS = ["ftsaf"]  # override: run only FT-SAF
gen_results = True
all_dbs = [NAME_REF_DB, NAME_FUTURE_DB]

if gen_results:
    start_time = time.time()  # record the start time

    for ptx_pathway in PTX_PATHWAYS:
        FILE_PATH_CASE_STUDIES_PTX = f"results/case_studies_ptx_{ptx_pathway}.pkl"
        FILE_PATH_CASE_STUDIES_LCA_PTX = f"results/case_studies_lca_ptx_{ptx_pathway}.pkl"

        all_totals = []
        all_totals_lca = []

        total_runs = len(all_dbs) * len(LOCATIONS) * len(scenarios)
        print(f"\n{'='*80}\nPATHWAY: {ptx_pathway}\n{'='*80}")
        print(f"Expected runs: {total_runs} ({len(all_dbs)} DBs ? {len(LOCATIONS)} locations ? {len(scenarios)} scenarios)")

        for db in all_dbs:
            print(f"\n{'='*80}\nDatabase: {db}\n{'='*80}")
            for country, iso2, lat, lon in LOCATIONS:
                print(f"\n***\n{country}\n***")
                cost_dict = COST_DICT.copy() if db == NAME_REF_DB else COST_DICT_FUTURE.copy()
                dict_ghg_impacts = DICT_GHG_IMPACTS.copy() if db == NAME_REF_DB else DICT_GHG_IMPACTS_FUTURE.copy()

                # get optimzation limits, with proper bounds
                dict_limits = ep.get_max_caps_regions()

                # get country-specific WACC/dr
                cost_dict['dr'] = ep.get_latest_avg_wacc(iso2)

                if db == NAME_REF_DB:
                    power_prices = ep.get_elect_prices(iso2)
                else:
                    avg_price = FUTURE_POWER_PRICES_DICT.get(iso2)
                    if avg_price is None:
                        avg_price = ep.get_elect_prices(iso2).mean()

                    factor_lower_night = 0.2
                    h_night_start = 19
                    h_night_end = 7

                    daily_prices = np.array([
                        avg_price * (1 - factor_lower_night)
                        if (h >= h_night_start or h < h_night_end)
                        else avg_price * (1 + factor_lower_night)
                        for h in range(24)
                    ])
                    power_prices = np.tile(daily_prices, 365)

                cell = ds_processed.sel(lat=lat, lon=lon, method="nearest")
                cf_pv = np.asarray(cell.cf_solar.values, dtype=float).copy()
                cf_wind = np.asarray(cell.cf_wind.values, dtype=float).copy()
                dac_el = np.asarray(cell.dac_el_kWh_per_kgCO2.values, dtype=float).copy()
                dac_heat = np.asarray(cell.dac_heat_kWh_per_kgCO2.values, dtype=float).copy()

                # And fill:
                df_data = pd.DataFrame(
                    data={
                        'pv_MW_array': cf_pv,
                        'wind_MW_array_on': cf_wind,
                        "grid_abs_price": power_prices,
                        "rev_inj": 0,
                    },
                    index=pd.date_range('1/1/{} 00:00'.format(2023), periods=8760, freq='h')
                )

                # Source local power grid mix GHG intensity
                df_data['ghg_impact'] = ep.get_activity_env_elect_from_dict(iso2, db=db)
                df_data['ghg_impact_cons'] = 0
                df_data['dac_el_kWh_per_kgCO2'] = dac_el
                df_data['dac_heat_kWh_per_kgCO2'] = dac_heat

                for scenario_name, kwargs in scenarios.items():
                    print(f"\nScenario: {scenario_name}")

                    df_data_use = df_data.copy()
                    dict_limits_use = dict_limits.copy()

                    if scenario_name == "grid_connected":
                        df_data_use["pv_MW_array"] = 0
                        df_data_use["wind_MW_array_on"] = 0
                    elif scenario_name == "off_grid":
                        dict_limits_use["max_grid_cap"] = 0
                        df_data_use["grid_abs_price"] = 0
                    elif scenario_name == "hybrid-green":
                        dict_limits_use["max_grid_cap"] = dict_limits_use["max_grid_cap"] * GREEN_GRID_CAP_FACTOR
                        kwargs = {**kwargs, "ghg_baseline_tco2_per_tprod": FOSSIL_BASELINE_TCO2[ptx_pathway]}

                    try:
                        # Run optimization
                        totals_cost_min, lca_results, __ = run_ptx_opt(
                            df_data_use, 1, 0, cost_dict, dict_ghg_impacts, dict_limits_use,
                            scenario_name=scenario_name,
                            loc_elect=iso2,
                            sec_db=db,
                            calc_all_lca_impacts=CALC_ALL_LCA_IMPACTS,
                            time_limit=SOLVER_TIME_LIMIT,
                            mip_gap=SOLVER_MIP_GAP,
                            int_feas_tol=SOLVER_INT_FEAS_TOL,
                            threads=SOLVER_THREADS,
                            ptx_pathway=ptx_pathway,
                            **kwargs
                        )
                    except Exception as e:
                        print(f"?? Optimization failed for {country}-{scenario_name}-{db}: {e}")
                        continue

                    if totals_cost_min is None:
                        print(f"?? No results for {country}-{scenario_name}-{db}")
                        continue

                    # Add identifying columns
                    totals_cost_min['country'] = country
                    totals_cost_min['iso2'] = iso2
                    totals_cost_min['scenario'] = scenario_name
                    totals_cost_min['db_name'] = db
                    totals_cost_min['ptx_pathway'] = ptx_pathway

                    all_totals.append(totals_cost_min)

                    if isinstance(lca_results, pd.DataFrame):
                        lca_df = lca_results.copy()

                        dup_cols = [name for name in lca_df.index.names if name in lca_df.columns]
                        if dup_cols:
                            lca_df = lca_df.drop(columns=dup_cols)

                        lca_df = lca_df.reset_index()

                        required_cols = ['country', 'iso2', 'scenario', 'category', 'contributor', 'db_name', 'year', 'ptx_pathway']
                        for col in required_cols:
                            if col not in lca_df.columns:
                                lca_df[col] = ""

                        lca_df['country'] = country
                        lca_df['iso2'] = iso2
                        lca_df['scenario'] = scenario_name
                        lca_df['db_name'] = db
                        lca_df['ptx_pathway'] = ptx_pathway

                        lca_df.set_index(required_cols, inplace=True)
                        all_totals_lca.append(lca_df)
                    else:
                        print(f"??  LCA not computed for {country} - {scenario_name} - {db}")

        # Save per-pathway results
        totals_cost_all = pd.concat(all_totals, ignore_index=True).set_index(['country', 'iso2', 'scenario', 'db_name', 'ptx_pathway'])
        totals_cost_all.to_pickle(FILE_PATH_CASE_STUDIES_PTX)

        if all_totals_lca:
            totals_cost_all_lca = pd.concat(all_totals_lca, ignore_index=False, axis=0)
            totals_cost_all_lca.to_pickle(FILE_PATH_CASE_STUDIES_LCA_PTX)
        else:
            print("??  No LCA results collected; skipping LCA pickle.")

    print(f"\nTotal time: {(time.time() - start_time)/3600:.2f} hours")
else:
    ptx_pathway = PTX_PATHWAYS[0]
    FILE_PATH_CASE_STUDIES_PTX = f"results/case_studies_ptx_{ptx_pathway}.pkl"
    FILE_PATH_CASE_STUDIES_LCA_PTX = f"results/case_studies_lca_ptx_{ptx_pathway}.pkl"
    with open(FILE_PATH_CASE_STUDIES_PTX, 'rb') as file:
        totals_cost_all = pickle.load(file)
    with open(FILE_PATH_CASE_STUDIES_LCA_PTX, 'rb') as file:
        totals_cost_all_lca = pickle.load(file)

totals_cost_all


In [ ]:
import os

# Check if results file exists
if os.path.exists(FILE_PATH_CASE_STUDIES_PTX):
    print(f"✓ Loading saved results from: {FILE_PATH_CASE_STUDIES_PTX}\n")
    
    # Load the saved results
    with open(FILE_PATH_CASE_STUDIES_PTX, 'rb') as file:
        totals_cost_all_loaded = pickle.load(file)
    
    print("="*80)
    print("SAVED RESULTS SUMMARY")
    print("="*80)
    print(f"Total cases completed: {len(totals_cost_all_loaded)}")
    print(f"Results shape: {totals_cost_all_loaded.shape}")
    print(f"\nIndex levels: {totals_cost_all_loaded.index.names}")
    print(f"\nColumns available ({len(totals_cost_all_loaded.columns)}): {list(totals_cost_all_loaded.columns[:15])}...")
    
    # Show breakdown by index levels
    if hasattr(totals_cost_all_loaded.index, 'levels'):
        print("\n" + "="*80)
        print("RESULTS BREAKDOWN:")
        print("="*80)
        for level_name in totals_cost_all_loaded.index.names:
            unique_vals = totals_cost_all_loaded.index.get_level_values(level_name).unique()
            print(f"\n{level_name}: {len(unique_vals)} unique values")
            print(f"  → {list(unique_vals)}")
    
    # Show first few rows
    print("\n" + "="*80)
    print("FIRST FEW RESULTS:")
    print("="*80)
    display(totals_cost_all_loaded.head())
    
else:
    print(f"⚠️  No saved results found at: {FILE_PATH_CASE_STUDIES_PTX}")
    print("\nTo generate results, run the optimization loop with gen_results=True")

# Check LCA results
print("\n" + "="*80)
print("CHECKING LCA RESULTS:")
print("="*80)

if os.path.exists(FILE_PATH_CASE_STUDIES_LCA_PTX):
    print(f"✓ LCA results found at: {FILE_PATH_CASE_STUDIES_LCA_PTX}")
    with open(FILE_PATH_CASE_STUDIES_LCA_PTX, 'rb') as file:
        totals_cost_all_lca_loaded = pickle.load(file)
    print(f"  LCA results shape: {totals_cost_all_lca_loaded.shape}")
    print(f"  Index levels: {totals_cost_all_lca_loaded.index.names}")
    if len(totals_cost_all_lca_loaded) > 0:
        print(f"  Columns: {list(totals_cost_all_lca_loaded.columns)}")
        display(totals_cost_all_lca_loaded.head())
else:
    print(f"⚠️  No LCA results found at: {FILE_PATH_CASE_STUDIES_LCA_PTX}")